<a href="https://colab.research.google.com/github/Sunn2x333/scalar_framework/blob/main/Planetary_BL3_Targets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
SFIB V12.1 Framework Simulation
Scalar Field as Information Bridge — Bobby Davis (2026)
https://doi.org/10.5281/zenodo.20533479

Version stamp: SFIB V12.1 Simulation — companion to Davis (2026), zenodo.20533479
               Extended kφ sector is EXPLORATORY and is clearly labeled as such.

─────────────────────────────────────────────────────────────────────────────
SIMULATION MODE  (set at top of file)
─────────────────────────────────────────────────────────────────────────────
  CORE_MODE = True
    • kG sector  : analytic Eq. 9, parameter-free
    • kφ sector  : ODE solver with CORE params only
                   (m_phi_squared, kappa, lambda_phi, k_phi_decay)
    • BL3 table  : β_φ |∇B|² L predictions
    Output: Tables A, C, D, Consistency Summary

  FULL_MODE = True  (superset of CORE)
    • All CORE outputs PLUS:
    • kφ sector extended: G_eff, σ_g, δY_p (BBN), gluon dressing
    • Table B: full stellar gradient profiles
    Note: extended constants are NOT constrained by V12.1 core claims.
          They are exploratory phenomenology for future cross-domain work.

─────────────────────────────────────────────────────────────────────────────
Architecture:
  - Single ordered pass: solar system (by AU), then stellar (by distance)
  - Each target produces both sector outputs in one sweep
  - kG / kφ sub-dicts clearly labeled; downstream analysis can pull either
  - X-regime classifier: SPACELIKE / NULL / TIMELIKE per target
  - Near-null flagging: targets within ε of X = 0 explicitly marked
─────────────────────────────────────────────────────────────────────────────
"""

# ═══════════════════════════════════════════════════════════════════════════
# ▶  SIMULATION MODE — set one or both True
# ═══════════════════════════════════════════════════════════════════════════
CORE_MODE = True    # kG sector + minimal kφ + BL3 (paper-anchored)
FULL_MODE = True    # extended kφ sector (exploratory, not V12.1 core claims)

# Near-null threshold: flag targets where X < NULL_EPS as "near-null candidates"
NULL_EPS  = 1e-20   # m⁻²

import numpy as np
from scipy.integrate import solve_ivp, trapezoid
from decimal import Decimal, getcontext, MAX_EMAX, MIN_EMIN

# ---------------------------------------------------------------------------
# Decimal precision
# ---------------------------------------------------------------------------
getcontext().prec = 28
getcontext().Emax = MAX_EMAX
getcontext().Emin = MIN_EMIN

def safe_exp_dec(x):
    if x > Decimal('700'):  return Decimal('inf')
    if x < Decimal('-700'): return Decimal('0')
    return x.exp()

def safe_exp(x):
    return np.exp(np.clip(x, -700, 700))

# ---------------------------------------------------------------------------
# Physical constants (SI)
# ---------------------------------------------------------------------------
C_LIGHT  = 299_792_458.0          # m/s
G_NEWTON = 6.674e-11              # m³ kg⁻¹ s⁻²
M_SUN    = 1.989e30               # kg
AU       = 1.495978707e11         # m  (IAU 2012)
MU_0     = 4 * np.pi * 1e-7      # T·m/A

# ---------------------------------------------------------------------------
# kG SECTOR constants — V12.1 §2.2, §4.4, §4.5
# These are the ONLY two free parameters in the kG sector.
# Both are extracted from independent data; neither is fit to a laboratory result.
# ---------------------------------------------------------------------------
K_G      = 2.27e17    # m   — from Jenkins–Fischbach orbital mechanics + decay data
TAU_BEAM = 888.0      # s   — Earth 1 AU beam baseline (NIST/ILL in-flight proton counting)

R_EARTH_AU = 1.000    # AU  (reference point for Eq. 9)

# ---------------------------------------------------------------------------
# kφ SECTOR constants
# ── CORE: appear in V12.1 Lagrangian or master integral directly ────────────
# ── EXTENDED [EXT]: exploratory dressing; not part of V12.1 core claims ────
# ---------------------------------------------------------------------------

# [CORE] Scalar field ODE parameters — derived from V12.1 Eq. 2–3
constants_scalar_core = {
    'm_phi_squared' : 1e-8,    # mφ² — sets Yukawa range ~0.2 mm
    'kappa'         : -1e-16,  # α coupling strength (stress-energy → field)
    'alpha_damping' : 1e-3,    # numerical regulariser for ODE solver
    'T_bound'       : 2.3e17,  # normalisation scale for stress-energy source
    'lambda_phi'    : 1e-2,    # φ⁴ self-coupling (chameleon potential)
}

# [EXT] Extended scalar sector — exploratory, not constrained by V12.1
constants_scalar_ext = {
    'k_wave'    : 1,
    'T_mu_nu'   : 1e9,
    'gamma_mu_nu': 1,
    'beta_comp' : 1e-5,
    'E_comp_th' : 1e31,
    'gamma_t'   : 0.1,
    'd2phi_dt2' : 1e-10,
    'M_hub'     : 1e53,
    'tau_exp'   : 4.3e17,
    'K_enhanced': 1,
    'gamma_comp': 2,
    'E_th'      : 1e31,
    'beta_bubble': 1e-5,
    'C_d'       : 1e-37,
    'm_phi'     : 1e-2,
    'beta_B'    : 1.0,
    'rho_0'     : 1e-25,
    'rho_Phi'   : 1e-9,
}
constants_scalar = {**constants_scalar_core, **constants_scalar_ext}

# [EXT] Gravity sector — exploratory modifications
constants_grav = {
    'G_0'       : 6.674e-11,
    'alpha_G'   : 1e-6,      # [EXT] G dressing from φ²
    'beta_fried': 1e-5,
    'rho_tot'   : 1e-27,
    'k_curv'    : 0,
    'Lambda_0'  : 1e-52,
    'a_grav'    : 1e12,
    'b_grav'    : 1e24,
    'e_g'       : 1e-5,
    'xi_metric' : 1,
    'rho_kin'   : 1e-20,
    'P_kin'     : 1e-20,
    'u_mu'      : 1,
    'g_mu_nu'   : 1,
    'v_rot_frac': 0.1,
    'f_5'       : 1e-5,
}

# [CORE/EXT mixed] EM sector — mu_0 is physical; K_pp, n_exp are exploratory
constants_em = {
    'mu_0'     : MU_0,       # [CORE] enters T_EM = B²/2μ₀
    'K_pp'     : 1e2,        # [EXT]
    'n_exp'    : 1.43,       # [EXT]
    'epsilon_0': 8.85e-12,
    'alpha_E'  : 1e-3,       # [EXT]
    'alpha_B'  : 1e-3,       # [EXT]
    'alpha_M'  : 1e-3,       # [EXT]
}

# [CORE] k_phi_decay — directly in the master integral (V12.1 Eq. 4)
# [EXT]  All others — phenomenological dressing not in V12.1 core
constants_decay = {
    'lambda_0_decay' : 1.13e-3,  # [CORE] free neutron decay rate
    'k_phi_decay'    : 1e-8,     # [CORE] kφ coupling in master integral
    'k_G'            : 2.6e-7,   # [CORE] kG in kφ-sector gravitational term
    # ── [EXT] below this line ──────────────────────────────────────────────
    'k_M'            : 0.1,
    'B_crit'         : 1e10,
    'a_m'            : 1e-3,
    'delta_m'        : 1e-3,
    'g_p'            : 1e-5,
    'Phi_0'          : 1e6,
    'k_gradiometry'  : 1e-8,
    'dP_constant'    : 1e-20,
    'P_total_constant': 1e-20,
    'alpha_s_0'      : 0.118,
    'k_S'            : 1e-20,
    'sigma_0'        : 0.4,
    'k_B_gluon'      : 0.5,      # [EXT] gluon dressing — not in V12.1 core
    'k_phi_gluon'    : 0.01,     # [EXT]
    'gamma_B'        : 1.5,      # [EXT]
    'B_th'           : 1e14,     # [EXT]
    'k_gluon'        : 0.01,     # [EXT]
    'tau_n'          : 880,
    'k_B_neutron'    : 8e-6,     # [EXT] tuned for ~1% beam shift (not EFT-derived)
    'm_n'            : 1.675e-27,
    'C_beta'         : 1e-30,
    'F_Z_Ee'         : 1.0,
    'p_e'            : 1e-3,
    'E_e'            : 0.5,
    'E_0'            : 0.782,
    'k_weak'         : 1e-3,
    'G_F'            : 1.166e-5,
}

# [EXT] Cosmological sector — BBN constraint check
constants_cosmo = {
    'k_phi_lens'    : Decimal('1.25e-6'),  # [EXT]
    'Y_p_true'      : Decimal('0.24'),
    'k_phi_dilation': 0.01,               # [EXT]
    'rho_bound'     : 1e31,
    'phi_golden'    : (1 + np.sqrt(5)) / 2,
    'n_hier'        : 10,
}
constants_univ = {
    'c_light'  : C_LIGHT,
    'q_proton' : 1.602e-19,
}

Msun_kg   = M_SUN
rho_scale = 1e3   # density unit bridge (see target catalogue)

# ---------------------------------------------------------------------------
# Target catalogue
# ---------------------------------------------------------------------------
# Solar system: distance_AU set → kG sector active; mass=0 (planetary body)
# Stellar:      distance_AU = None → kG sector inactive

targets = {
    # ── Solar system (kG sector active) ─────────────────────────────────────
    'Mercury perihelion': {
        'mass': 0.0, 'radius': 2440, 'rho_c': 5e-9, 'B': 1e-4,
        'P_rot': 5.06e6, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        # 46.0e9 m = 0.30750 AU (NASA); paper rounds to 0.307 AU → 776.7 vs 776.6 s
        'distance_AU': 46.0e9 / 1.495978707e11,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Mercury mean': {
        'mass': 0.0, 'radius': 2440, 'rho_c': 5e-9, 'B': 1e-4,
        'P_rot': 5.06e6, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 0.387,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Mercury aphelion': {
        'mass': 0.0, 'radius': 2440, 'rho_c': 5e-9, 'B': 1e-4,
        'P_rot': 5.06e6, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 0.467,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Venus mean': {
        'mass': 0.0, 'radius': 6052, 'rho_c': 5e-9, 'B': 1e-5,
        'P_rot': 2.1e7, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 0.723,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Earth mean': {
        'mass': 0.0, 'radius': 6371, 'rho_c': 5e-9, 'B': 5e-5,
        'P_rot': 86400, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 1.000,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Moon': {
        'mass': 0.0, 'radius': 1737, 'rho_c': 5e-9, 'B': 1e-7,
        'P_rot': 2.36e6, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 1.003,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Mars mean': {
        'mass': 0.0, 'radius': 3390, 'rho_c': 5e-9, 'B': 1e-6,
        'P_rot': 88620, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 1.523,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Jupiter mean': {
        'mass': 0.0, 'radius': 71492, 'rho_c': 1.33e3, 'B': 4e-4,
        'P_rot': 35730, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 5.203,
        'type': 'Solar System', 'use_sun_phi': True,
    },
    'Deep space (100 AU)': {
        'mass': 0.0, 'radius': 1, 'rho_c': 1e-25, 'B': 1e-10,
        'P_rot': 1e8, 'dBdt': 0.0, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': 100.0,
        'type': 'Solar System', 'use_sun_phi': True,
    },

    # ── Main sequence ────────────────────────────────────────────────────────
    'Sun': {
        'mass': 1.0, 'radius': 695700, 'rho_c': 162, 'B': 2,
        'P_rot': 2.16e6, 'dBdt': 1e-10, 't_age': 1.45e17, 'distance': 0.0,
        'distance_AU': None, 'type': 'Main-Sequence', 'use_sun_phi': False,
    },
    'Alpha Centauri A': {
        'mass': 1.0788, 'radius': 1.2175*695700, 'rho_c': 150, 'B': 5,
        'P_rot': 2e6, 'dBdt': 1e-10, 't_age': 1.7e17, 'distance': 4.37,
        'distance_AU': None, 'type': 'Main-Sequence', 'use_sun_phi': False,
    },
    'Tau Ceti': {
        'mass': 0.78, 'radius': 0.79*695700, 'rho_c': 200, 'B': 1,
        'P_rot': 2.9e6, 'dBdt': 1e-10, 't_age': 1.8e17, 'distance': 11.9,
        'distance_AU': None, 'type': 'Main-Sequence', 'use_sun_phi': False,
    },
    'Epsilon Eridani': {
        'mass': 0.82, 'radius': 0.735*695700, 'rho_c': 210, 'B': 3,
        'P_rot': 4.9e5, 'dBdt': 1e-10, 't_age': 8e16, 'distance': 10.5,
        'distance_AU': None, 'type': 'Main-Sequence', 'use_sun_phi': False,
    },

    # ── Subgiants / giants ───────────────────────────────────────────────────
    'Beta Hydri': {
        'mass': 1.08, 'radius': 1.81*695700, 'rho_c': 50, 'B': 7,
        'P_rot': 4e6, 'dBdt': 1e-10, 't_age': 2.1e17, 'distance': 24.3,
        'distance_AU': None, 'type': 'Subgiant', 'use_sun_phi': False,
    },
    'Delta Pavonis': {
        'mass': 1.12, 'radius': 1.22*695700, 'rho_c': 140, 'B': 6,
        'P_rot': 3e6, 'dBdt': 1e-10, 't_age': 2e17, 'distance': 19.9,
        'distance_AU': None, 'type': 'Subgiant', 'use_sun_phi': False,
    },
    'Aldebaran': {
        'mass': 1.16, 'radius': 44.2*695700, 'rho_c': 0.08, 'B': 1,
        'P_rot': 5.2e7, 'dBdt': 1e-10, 't_age': 3.5e17, 'distance': 65.3,
        'distance_AU': None, 'type': 'Red Giant', 'use_sun_phi': False,
    },

    # ── White dwarfs ─────────────────────────────────────────────────────────
    'Sirius B': {
        'mass': 1.02, 'radius': 5800, 'rho_c': 2e6, 'B': 1e5,
        'P_rot': 1e6, 'dBdt': 1e-10, 't_age': 4e15, 'distance': 8.6,
        'distance_AU': None, 'type': 'White Dwarf', 'use_sun_phi': False,
    },
    '40 Eridani B': {
        'mass': 0.573, 'radius': 7680, 'rho_c': 8e5, 'B': 4e3,
        'P_rot': 3.5e4, 'dBdt': 1e-10, 't_age': 5.5e16, 'distance': 16.3,
        'distance_AU': None, 'type': 'White Dwarf', 'use_sun_phi': False,
    },

    # ── Neutron stars ────────────────────────────────────────────────────────
    'Crab Pulsar': {
        'mass': 1.4, 'radius': 10, 'rho_c': 1e14, 'B': 1e8,
        'P_rot': 0.033, 'dBdt': 1e-10, 't_age': 3.156e10, 'distance': 6500,
        'distance_AU': None, 'type': 'Neutron Star', 'use_sun_phi': False,
    },
    'PSR J0030+0451': {
        # NICER-measured radius; cited in V12.1 Table 6
        'mass': 1.34, 'radius': 13, 'rho_c': 8e13, 'B': 1e8,
        'P_rot': 0.00485, 'dBdt': 1e-10, 't_age': 5e17, 'distance': 1000,
        'distance_AU': None, 'type': 'Neutron Star', 'use_sun_phi': False,
    },

    # ── Magnetars ────────────────────────────────────────────────────────────
    'SGR 1806-20': {
        'mass': 1.5, 'radius': 12, 'rho_c': 8e13, 'B': 1e10,
        'P_rot': 7.5, 'dBdt': 1e-10, 't_age': 1e12, 'distance': 42000,
        'distance_AU': None, 'type': 'Magnetar', 'use_sun_phi': False,
    },
    '1E 1841-045': {
        'mass': 1.3, 'radius': 11, 'rho_c': 9e13, 'B': 7e10,
        'P_rot': 11.8, 'dBdt': 1e-10, 't_age': 1e12, 'distance': 28000,
        'distance_AU': None, 'type': 'Magnetar', 'use_sun_phi': False,
    },
    '4U 0142+61': {
        'mass': 1.45, 'radius': 13, 'rho_c': 7e13, 'B': 1.3e10,
        'P_rot': 8.69, 'dBdt': 1e-10, 't_age': 5e11, 'distance': 13000,
        'distance_AU': None, 'type': 'Magnetar', 'use_sun_phi': False,
    },

    # ── Black holes ──────────────────────────────────────────────────────────
    'Cygnus X-1': {
        # Stellar-mass BH; Schwarzschild radius ≈ 88 km for 21.2 M☉
        'mass': 21.2, 'radius': 88, 'rho_c': 1e15, 'B': 1e6,
        'P_rot': 5e4, 'dBdt': 1e-10, 't_age': 5e15, 'distance': 7200,
        'distance_AU': None, 'type': 'Black Hole', 'use_sun_phi': False,
    },
    'TON 618': {
        'mass': 6.6e10, 'radius': 1e9, 'rho_c': 1e10, 'B': 1e4,
        'P_rot': 1e8, 'dBdt': 1e-10, 't_age': 4e17, 'distance': 1.08e10,
        'distance_AU': None, 'type': 'Black Hole', 'use_sun_phi': False,
    },
}

# Paper Table 4 reference values — V12.1 §4.5
# Used for Table A relative-difference column and consistency summary
PAPER_TAU_REF = {
    'Mercury perihelion' : 776.7,
    'Mercury mean'       : 818.5,
    'Mercury aphelion'   : 842.7,
    'Venus mean'         : 876.1,
    'Earth mean'         : 888.0,
    'Moon'               : 888.0,
    'Mars mean'          : 895.6,
    'Jupiter mean'       : None,
    'Deep space (100 AU)': 901.0,
}

# ---------------------------------------------------------------------------
# kG SECTOR — analytic Eq. 9 (V12.1 §4.5)
# ---------------------------------------------------------------------------

def tau_n_analytic(r_AU):
    """
    Eq. 9:  τ_n(r) = τ_beam / [1 + kG · (GM☉/c²r² − GM☉/c²r⊕²)]
    Parameter-free given kG extracted from Jenkins–Fischbach.
    """
    r_m   = r_AU * AU
    r_e_m = R_EARTH_AU * AU
    GM    = G_NEWTON * M_SUN
    delta = (GM / C_LIGHT**2) * (1.0/r_m**2 - 1.0/r_e_m**2)
    return TAU_BEAM / (1.0 + K_G * delta)


def compute_kG_sector(params):
    r_AU = params.get('distance_AU')
    if r_AU is None:
        return {'tau_analytic': None, 'delta_tau_analytic': None, 'kG_active': False}
    tau = tau_n_analytic(r_AU)
    return {
        'tau_analytic'       : tau,
        'delta_tau_analytic' : tau - TAU_BEAM,
        'kG_active'          : True,
    }

# ---------------------------------------------------------------------------
# kφ SECTOR — ODE scalar field solver (V12.1 §3.2)
# ---------------------------------------------------------------------------

def hierarchical_bubble(N_levels=80, golden=(1+np.sqrt(5))/2, base_nabla_sq=6e-7):
    return base_nabla_sq * sum(1 / golden**(2*i) for i in range(N_levels))


def compute_phi_for_target(params):
    """Solve the scalar field ODE; return averaged quantities needed by both modes."""
    radius_m   = max(params['radius'] * 1000, 1e3)
    sigma_r    = radius_m / 3
    rho_c_kgm3 = params['rho_c'] * rho_scale

    B    = params['B'] / 1e4
    T_EM = B**2 / (2 * constants_em['mu_0'])

    def dphi_dr(current_r, state):
        phi, phi_p = state
        rho_r = rho_c_kgm3 * np.exp(-current_r**2 / (2*sigma_r**2))
        T     = rho_r * C_LIGHT**2 + T_EM
        d2phi = (-constants_scalar_core['m_phi_squared'] * phi
                 - constants_scalar_core['kappa'] * T / constants_scalar_core['T_bound']
                 - constants_scalar_core['alpha_damping'] * phi_p)
        return [phi_p, d2phi]

    sol = solve_ivp(dphi_dr, [0, radius_m], [1e-3, 0.0],
                    method='LSODA', rtol=1e-5, atol=1e-8,
                    max_step=radius_m/100)

    r_sol         = sol.t
    phi_sol       = sol.y[0]
    phi_prime_sol = sol.y[1]
    max_r         = r_sol[-1]

    avg_phi     = trapezoid(phi_sol,          r_sol) / max_r
    avg_grad_sq = trapezoid(phi_prime_sol**2, r_sol) / max_r

    eff_nabla_sq = hierarchical_bubble(base_nabla_sq=6e-7 * (rho_c_kgm3 / 1e5))
    avg_grad_sq *= (eff_nabla_sq / 6e-7)

    E_phi = (0.5 * phi_prime_sol**2
             - (constants_em['K_pp'] / 8) * phi_sol**4
             - (constants_scalar_core['lambda_phi'] / 2) * phi_sol**2)
    E_phi_avg = trapezoid(E_phi, r_sol) / max_r

    rho_r_sol = rho_c_kgm3 * np.exp(-r_sol**2 / (2*sigma_r**2))
    T_r_sol   = rho_r_sol * C_LIGHT**2 + T_EM * np.ones_like(r_sol)

    return (r_sol, phi_sol, phi_prime_sol, T_r_sol, rho_r_sol,
            avg_phi, avg_grad_sq, E_phi_avg)


def compute_kphi_core(avg_grad_sq, params):
    """
    [CORE] Minimal kφ sector: only master integral + Sargent amplification.
    λ_eff = λ₀(1 + k_phi_decay · ⟨|∇φ|²⟩)  — V12.1 Eq. 4
    Returns τ_kφ (s) and fractional shift δτ/τ.
    """
    lam0    = constants_decay['lambda_0_decay']
    kphi    = constants_decay['k_phi_decay']
    lam_eff = lam0 * (1.0 + kphi * avg_grad_sq)
    tau     = 1.0 / lam_eff if lam_eff > 0 else None
    dtau    = (tau - TAU_BEAM) / TAU_BEAM if tau else None
    return {'tau_core': tau, 'dtau_core': dtau, 'lambda_core': lam_eff}


def compute_kphi_extended(avg_phi, avg_grad_sq, E_phi_avg, params,
                           T_r_sol, rho_r_sol, r_sol):
    """
    [EXTENDED — EXPLORATORY] Full phenomenological kφ dressing.
    G_eff, G_EM, σ_g, gluon correction, BBN δY_p.
    NOT part of V12.1 core claims; constants are not EFT-constrained.
    """
    if not FULL_MODE:
        return {}

    max_r  = r_sol[-1]
    c      = C_LIGHT
    M_kg   = params['mass'] * Msun_kg
    B      = params['B'] / 1e4

    G_eff  = constants_grav['G_0'] + constants_grav['alpha_G'] * avg_phi**2
    G_EM   = G_eff * (1 - constants_scalar_ext['beta_B'] * B**2 * avg_phi**2)

    Phi_grav = (constants_grav['G_0'] * M_kg
                / max(params['radius']*1000, 1.0)) / c**2

    lam0  = constants_decay['lambda_0_decay']
    lam27 = lam0 * (1
                    + constants_decay['k_G']        * Phi_grav
                    + constants_decay['k_M']        * (B**2 / constants_decay['B_crit']**2)
                    + constants_decay['k_phi_decay'] * avg_grad_sq)

    sig_g  = (constants_decay['sigma_0']
              * (1
                 + constants_decay['k_B_gluon']  * (B**2 / constants_decay['B_crit']**2)
                 + constants_decay['k_phi_gluon'] * avg_grad_sq)
              * safe_exp(constants_decay['gamma_B'] * (B / constants_decay['B_th'])))

    lam51 = lam0 * (1
                    + constants_decay['k_phi_decay'] * avg_grad_sq
                    + constants_decay['k_gluon']     * sig_g)
    if params['type'] in ('Neutron Star', 'Magnetar'):
        lam51 *= (1 + constants_decay['k_B_neutron'] * B**2)

    # Compact-object flag for τ_kφ — λ_eff >> 1/s is non-physical for free neutrons
    tau_ext = None
    tau_flag = ""
    if lam51 > 0:
        tau_val = 1.0 / lam51
        if tau_val < 1e-3:
            tau_flag = "[non-physical: extreme env]"
        else:
            tau_ext = tau_val

    # BBN δY_p [EXT]
    ds              = Decimal(str(params['radius'] * 1000))
    avg_grad_sq_dec = Decimal(str(avg_grad_sq))
    exp_arg         = constants_cosmo['k_phi_lens'] * avg_grad_sq_dec * ds
    ln_Y            = constants_cosmo['Y_p_true'].ln() + exp_arg
    Y_obs           = float(safe_exp_dec(ln_Y))
    if params['type'] in ('Neutron Star', 'Magnetar'):
        Y_obs = float(constants_cosmo['Y_p_true'])
    delta_Yp = Y_obs - float(constants_cosmo['Y_p_true'])

    return {
        'G_eff'         : G_eff,
        'G_EM'          : G_EM,
        'Phi_grav'      : Phi_grav,
        'lambda_eff_27' : lam27,
        'lambda_eff_51' : lam51,
        'sigma_g'       : sig_g,
        'tau_ext'       : tau_ext,
        'tau_ext_flag'  : tau_flag,
        'Y_observed'    : Y_obs,
        'delta_Yp'      : delta_Yp,
    }


def compute_tau_ode_crosscheck(avg_grad_sq, params):
    """
    ODE cross-check for solar system targets.
    kG analytic + small kφ correction from ODE gradient.
    Expected diff ≈ 0 at low density (validates sector independence).
    """
    r_AU = params.get('distance_AU')
    if r_AU is None:
        return None
    tau_kG  = tau_n_analytic(r_AU)
    delta   = constants_decay['k_phi_decay'] * avg_grad_sq
    return tau_kG * (1.0 - delta)


def classify_X_regime(avg_grad_sq, params):
    """
    Classify the covariant invariant X = ∂_μφ ∂^μφ.
    Static approximation (V12.1 §3.4): X ≈ |∇φ|² for lab/stellar sources.
    Returns (regime_str, X_value, near_null_flag).
    """
    X = float(avg_grad_sq)
    near_null = (0 < X < NULL_EPS)
    if X <= 0:
        return 'NULL', 0.0, False
    if near_null:
        return 'SPACELIKE', X, True
    return 'SPACELIKE', X, False

# ---------------------------------------------------------------------------
# BL3 trim coil sweep — V12.1 §8.1
# ---------------------------------------------------------------------------
BETA_PHI = 0.097  # m/T²  — calibrated from V12.1 Eq. 15; same value predicts §8.3 fringe shift

def bl3_tau(grad_B, L):
    """δτ/τ = β_φ |∇B|² L  → τ_pred, frac_shift"""
    frac = BETA_PHI * grad_B**2 * L
    return TAU_BEAM * (1.0 - frac), frac

BL3_SWEEP = [
    # (label,                 |∇B| T/m, L m,  is_BL2_ref)
    ('BL3 zero gradient'  ,  0.000, 10.0, False),
    ('BL3 low  0.01 T/m'  ,  0.010, 10.0, False),
    ('BL3 mid  0.05 T/m'  ,  0.050, 10.0, False),
    ('BL3 nom  0.10 T/m'  ,  0.100, 10.0, False),
    ('BL3 high 0.20 T/m'  ,  0.200, 10.0, False),
    ('BL3 max  0.50 T/m'  ,  0.500, 10.0, False),
    ('BL2 reference'       ,  0.070,  8.0, True ),  # BL2 nominal geometry estimate
    # Detector-position scan (fixed |∇B| = 0.10 T/m)
    ('BL3 L=2 m'           ,  0.100,  2.0, False),
    ('BL3 L=5 m'           ,  0.100,  5.0, False),
    ('BL3 L=10 m'          ,  0.100, 10.0, False),
    ('BL3 L=15 m'          ,  0.100, 15.0, False),
]

# ---------------------------------------------------------------------------
# Main simulation loop
# ---------------------------------------------------------------------------

def run():
    results = {}

    solar   = {k: v for k,v in targets.items() if v['distance_AU'] is not None}
    stellar = {k: v for k,v in targets.items() if v['distance_AU'] is None}
    ordered = (sorted(solar.items(),   key=lambda x: x[1]['distance_AU'])
             + sorted(stellar.items(), key=lambda x: x[1]['distance']))

    for name, params in ordered:
        params['name'] = name
        print(f"  {name:<32} [{params['type']}]")

        kG_res   = compute_kG_sector(params)

        (r_sol, phi_sol, phi_prime_sol, T_r_sol, rho_r_sol,
         avg_phi, avg_grad_sq, E_phi_avg) = compute_phi_for_target(params)

        core_res = compute_kphi_core(avg_grad_sq, params)
        ext_res  = compute_kphi_extended(avg_phi, avg_grad_sq, E_phi_avg,
                                          params, T_r_sol, rho_r_sol, r_sol)
        tau_ode  = compute_tau_ode_crosscheck(avg_grad_sq, params)
        regime, X_val, near_null = classify_X_regime(avg_grad_sq, params)

        results[name] = {
            **kG_res,
            **core_res,
            **ext_res,
            'tau_ode'      : tau_ode,
            'avg_phi'      : avg_phi,
            'avg_grad_sq'  : avg_grad_sq,
            'E_phi_avg'    : E_phi_avg,
            'X_regime'     : regime,
            'X_value'      : X_val,
            'near_null'    : near_null,
            'type'         : params['type'],
            'distance'     : params['distance'],
            'distance_AU'  : params.get('distance_AU'),
        }

    return results, ordered

# ---------------------------------------------------------------------------
# Output helpers
# ---------------------------------------------------------------------------
W   = 140
SEP  = "─" * W
SEP2 = "═" * W

def fv(val, spec='.3e'):
    return "    N/A" if val is None else format(val, spec)


# ---------------------------------------------------------------------------
# TABLE A — kG sector solar system predictions
# ---------------------------------------------------------------------------

def print_table_A(results, ordered):
    print(); print(SEP2)
    print("  TABLE A — kG SECTOR: Solar System Neutron Lifetime Predictions  [CORE]")
    print(f"  Eq. 9 (V12.1 §4.5)  |  kG = {K_G:.2e} m  |  τ_beam = {TAU_BEAM} s  |  AU = 1.495978707×10¹¹ m (IAU 2012)")
    print("  MESSENGER measurement: τ_n = 780 ± 90 s at Mercury (Wilson et al. 2020)")
    print(SEP2)

    hdr = (f"  {'Target':<22} {'AU':>8} {'τ_analytic (s)':>16} {'Δτ_Earth (s)':>14} "
           f"{'τ_ODE chk (s)':>15} {'Paper (s)':>11} {'|Δ| (s)':>9} {'Δ/τ (%)':>9}")
    print(hdr); print(SEP)

    residuals = []
    for name, _ in ordered:
        r = results[name]
        if not r['kG_active']:
            continue
        tau_a = r['tau_analytic']
        tau_o = r['tau_ode']
        pref  = PAPER_TAU_REF.get(name)
        abs_diff = abs(tau_a - pref) if pref else None
        rel_diff = 100.0 * abs_diff / pref if abs_diff else None
        if abs_diff is not None:
            residuals.append((name, abs_diff, rel_diff))

        diff_str  = f"{abs_diff:>9.2f}" if abs_diff is not None else "        —"
        rdiff_str = f"{rel_diff:>9.4f}" if rel_diff is not None else "        —"
        pref_str  = f"{pref:>11.1f}" if pref is not None else "          —"
        flag = "  ◄ MESSENGER 780±90 s" if name == 'Mercury perihelion' else ""

        print(f"  {name:<22} {r['distance_AU']:>8.4f} {tau_a:>16.3f} "
              f"{r['delta_tau_analytic']:>14.3f} {fv(tau_o, '.3f'):>15} "
              f"{pref_str} {diff_str} {rdiff_str}{flag}")

    print(SEP)
    if residuals:
        worst = max(residuals, key=lambda x: x[1])
        rms   = np.sqrt(np.mean([x[1]**2 for x in residuals]))
        print(f"  Max |τ_analytic − τ_paper| = {worst[1]:.3f} s  ({worst[2]:.4f}%)  at {worst[0]}")
        print(f"  RMS residual across {len(residuals)} solar system entries = {rms:.3f} s")
    print("  ◄ SFIB Mercury perihelion is the ONLY hypothesis below MESSENGER central value.")
    print("    Beam-everywhere +108 s  |  Bottle-everywhere +97 s  |  PDG avg +99 s")


# ---------------------------------------------------------------------------
# TABLE B — kφ extended sector stellar profiles (FULL_MODE only)
# ---------------------------------------------------------------------------

def print_table_B(results, ordered):
    if not FULL_MODE:
        return
    print(); print(SEP2)
    print("  TABLE B — kφ SECTOR: Stellar & Compact Object Gradient Profiles  [EXTENDED — EXPLORATORY]")
    print("  WARNING: Extended constants (k_M, k_B_gluon, k_B_neutron, etc.) are NOT constrained")
    print("           by V12.1 core claims. This table is for cross-domain exploration only.")
    print(SEP2)
    hdr = (f"  {'Target':<22} {'Type':<13} {'Dist (ly)':>9} {'⟨|∇φ|²⟩':>14} "
           f"{'τ_core (s)':>11} {'τ_ext (s)':>12} {'λ_eff_51':>12} {'δY_p':>12}")
    print(hdr); print(SEP)

    for name, _ in ordered:
        r = results[name]
        if r['kG_active']:
            continue
        tau_c    = r.get('tau_core')
        tau_e    = r.get('tau_ext')
        tau_flag = r.get('tau_ext_flag', '')
        lam51    = r.get('lambda_eff_51', None)
        dYp      = r.get('delta_Yp', None)

        tau_e_str = f"{tau_e:>12.1f}" if tau_e else f"  {tau_flag:<20}"

        print(f"  {name:<22} {r['type']:<13} {r['distance']:>9.1f} "
              f"{r['avg_grad_sq']:>14.3e} {fv(tau_c, '.1f'):>11} "
              f"{tau_e_str} {fv(lam51, '.3e'):>12} {fv(dYp, '.3e'):>12}")
    print(SEP)


# ---------------------------------------------------------------------------
# TABLE C — BL3 trim coil sweep
# ---------------------------------------------------------------------------

def print_table_C():
    print(); print(SEP2)
    print("  TABLE C — BL3 TRIM COIL SWEEP PREDICTIONS  [CORE]  (V12.1 §8.1)")
    print(f"  β_φ = {BETA_PHI} m/T²  (same β_φ independently predicts §8.3 double-slit fringe shift)")
    print(f"  δτ/τ = β_φ |∇B|² L  |  Gradient-reversal parity: same δτ for ±|∇B|")
    print(SEP2)
    hdr = (f"  {'Configuration':<26} {'|∇B| T/m':>10} {'L (m)':>7} "
           f"{'δτ/τ':>10} {'δτ (s)':>9} {'τ_pred (s)':>12} {'Note':<18}")
    print(hdr); print(SEP)

    for label, grad, L, is_bl2 in BL3_SWEEP:
        tau_p, frac = bl3_tau(grad, L)
        dtau = tau_p - TAU_BEAM
        note = "BL2 estimate" if is_bl2 else ("SFIB null ← X=0" if grad == 0 else "")
        print(f"  {label:<26} {grad:>10.3f} {L:>7.1f} "
              f"{frac:>10.5f} {dtau:>9.3f} {tau_p:>12.3f} {note:<18}")

    tau_nom, frac_nom = bl3_tau(0.10, 10.0)
    tau_bl2, _        = bl3_tau(0.07,  8.0)
    print(SEP)
    print(f"  SFIB signature: BL3 (nom) = {tau_nom:.2f} s  <  BL2 est = {tau_bl2:.2f} s  <  beam baseline {TAU_BEAM} s")
    print(f"  Required experimental precision to resolve 1% SFIB shift: σ_stat+sys ≲ 0.3%")
    print(f"  (Shift scales as |∇B|²·L — a controllable dial built into BL3 trim coil system)")


# ---------------------------------------------------------------------------
# TABLE D — X-regime classification
# ---------------------------------------------------------------------------

def print_table_D(results, ordered):
    print(); print(SEP2)
    print("  TABLE D — X-REGIME CLASSIFICATION  [CORE]  (V12.1 §3.4)")
    print(f"  X = ∂_μφ ∂^μφ  |  NULL_EPS threshold = {NULL_EPS:.1e} m⁻²")
    print("  SPACELIKE (X>0): λ_eff > λ₀  |  NULL (X=0): λ_eff = λ₀  |  TIMELIKE (X<0): λ_eff < λ₀")
    print(SEP2)
    hdr = (f"  {'Target':<25} {'Type':<13} {'Regime':<11} {'X (m⁻²)':>14} "
           f"{'Effect on λ':>18} {'Near-null?':>12}")
    print(hdr); print(SEP)

    counts = {'SPACELIKE': 0, 'NULL': 0, 'TIMELIKE': 0}
    near_nulls = []

    for name, _ in ordered:
        r = results[name]
        regime = r['X_regime']
        counts[regime] += 1
        nn_str = "★ NEAR-NULL" if r['near_null'] else ""
        if r['near_null']:
            near_nulls.append(name)

        effect = {'SPACELIKE': 'Enhanced  (λ > λ₀)',
                  'NULL'      : 'Neutral   (λ = λ₀)',
                  'TIMELIKE'  : 'Suppressed (λ < λ₀)'}.get(regime, '???')

        print(f"  {name:<25} {r['type']:<13} {regime:<11} {r['X_value']:>14.3e} "
              f"{effect:>18} {nn_str:>12}")

    print(SEP)
    print(f"  Regime count — SPACELIKE: {counts['SPACELIKE']}  |  NULL: {counts['NULL']}  "
          f"|  TIMELIKE: {counts['TIMELIKE']}")
    if near_nulls:
        print(f"  Near-null candidates (0 < X < {NULL_EPS:.0e}): {', '.join(near_nulls)}")
    else:
        print(f"  No near-null candidates found at NULL_EPS = {NULL_EPS:.0e} m⁻²")
    print("  (Timelike sector — BBN, cosmological evolution — not present in static targets)")


# ---------------------------------------------------------------------------
# CONSISTENCY SUMMARY
# ---------------------------------------------------------------------------

def print_consistency_summary(results, ordered):
    print(); print(SEP2)
    print("  CONSISTENCY SUMMARY")
    print(SEP2)

    # ── kG sector ────────────────────────────────────────────────────────────
    solar_residuals = []
    for name, _ in ordered:
        r = results[name]
        if not r['kG_active']:
            continue
        pref = PAPER_TAU_REF.get(name)
        if pref:
            solar_residuals.append(abs(r['tau_analytic'] - pref))

    max_res = max(solar_residuals) if solar_residuals else None
    rms_res = np.sqrt(np.mean(np.array(solar_residuals)**2)) if solar_residuals else None

    merc = results.get('Mercury perihelion', {})
    tau_merc = merc.get('tau_analytic')
    if tau_merc:
        print(f"  [kG]  Mercury perihelion:  τ = {tau_merc:.2f} s  |  MESSENGER = 780 ± 90 s  "
              f"|  residual = {abs(tau_merc - 780):.2f} s")
    print(f"  [kG]  Max |τ_analytic − τ_paper| over solar system entries: "
          f"{max_res:.3f} s" if max_res else "  [kG]  No paper reference entries found.")
    print(f"  [kG]  RMS residual: {rms_res:.3f} s" if rms_res else "")

    tau_ds = results.get('Deep space (100 AU)', {}).get('tau_analytic')
    if tau_ds:
        print(f"  [kG]  Deep space τ₀ = {tau_ds:.2f} s  (→ asymptote ≈ 901 s; "
              f"every Earth measurement shifted below this by kG at 1 AU)")

    tau_v = results.get('Venus mean', {}).get('tau_analytic')
    if tau_v:
        print(f"  [kG]  Venus coincidence: τ_Venus = {tau_v:.1f} s ≈ UCN bottle (~877 s)  "
              f"[gradient, not confinement]")

    # ── BL3 ──────────────────────────────────────────────────────────────────
    tau_nom,  frac_nom  = bl3_tau(0.10, 10.0)
    tau_zero, _         = bl3_tau(0.00, 10.0)
    tau_bl2,  _         = bl3_tau(0.07,  8.0)
    print(f"\n  [BL3] Nominal prediction (0.10 T/m, 10 m): τ = {tau_nom:.3f} s  "
          f"(δτ/τ = {frac_nom:.4f} ≈ {100*frac_nom:.2f}%)")
    print(f"  [BL3] Zero-gradient null:  τ = {tau_zero:.3f} s  (δτ = 0 exactly — SFIB null surface)")
    print(f"  [BL3] BL3 ({tau_nom:.1f} s) < BL2 est ({tau_bl2:.1f} s) < beam baseline ({TAU_BEAM} s)  ✓")

    # ── X-regime count ───────────────────────────────────────────────────────
    counts = {'SPACELIKE': 0, 'NULL': 0, 'TIMELIKE': 0}
    for name, _ in ordered:
        counts[results[name]['X_regime']] += 1
    print(f"\n  [X]   Regime distribution: SPACELIKE={counts['SPACELIKE']}  "
          f"NULL={counts['NULL']}  TIMELIKE={counts['TIMELIKE']}")
    print(f"  [X]   TIMELIKE regime (cosmological / BBN) not populated in static target set — "
          f"consistent with §9.")

    # ── Mode label ───────────────────────────────────────────────────────────
    mode_str = ("CORE + FULL (extended kφ active)"
                if FULL_MODE else "CORE only (paper-anchored)")
    print(f"\n  Simulation mode: {mode_str}")
    print(f"  Version stamp: SFIB V12.1 Simulation — companion to Davis (2026), zenodo.20533479")
    print(f"                 Extended kφ sector: EXPLORATORY — not V12.1 core claims")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == '__main__':
    import datetime
    ts = datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

    print()
    print("=" * W)
    print("  SFIB V12.1 Framework Simulation — Scalar Field as Information Bridge")
    print(f"  Bobby Davis 2026  |  https://doi.org/10.5281/zenodo.20533479")
    print(f"  Run: {ts}  |  Mode: {'CORE+FULL' if FULL_MODE else 'CORE'}")
    print(f"  kG = {K_G:.2e} m  |  β_φ = {BETA_PHI} m/T²  |  τ_beam = {TAU_BEAM} s  |  NULL_EPS = {NULL_EPS:.0e} m⁻²")
    print("=" * W)
    print()

    results, ordered = run()

    print_table_A(results, ordered)
    print_table_B(results, ordered)
    print_table_C()
    print_table_D(results, ordered)
    print_consistency_summary(results, ordered)

    print()
    print("=" * W)
    print("  Done.")
    print("=" * W)


  SFIB V12.1 Framework Simulation — Scalar Field as Information Bridge
  Bobby Davis 2026  |  https://doi.org/10.5281/zenodo.20533479
  Run: 2026-06-22 06:06 UTC  |  Mode: CORE+FULL
  kG = 2.27e+17 m  |  β_φ = 0.097 m/T²  |  τ_beam = 888.0 s  |  NULL_EPS = 1e-20 m⁻²

  Mercury perihelion               [Solar System]
  Mercury mean                     [Solar System]
  Mercury aphelion                 [Solar System]
  Venus mean                       [Solar System]
  Earth mean                       [Solar System]
  Moon                             [Solar System]
  Mars mean                        [Solar System]
  Jupiter mean                     [Solar System]
  Deep space (100 AU)              [Solar System]
  Sun                              [Main-Sequence]
  Alpha Centauri A                 [Main-Sequence]
  Sirius B                         [White Dwarf]
  Epsilon Eridani                  [Main-Sequence]
  Tau Ceti                         [Main-Sequence]
  40 Eridani B              